In [1]:
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.datasets import fetch_covtype
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, log_loss
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch import Tensor

import torch._dynamo
import torch.utils.benchmark as benchmark
from torch.optim import Optimizer
from torchinfo import summary
from torch.autograd import Function
import torch._inductor.metrics as metrics

In [2]:
torch._dynamo.config.cache_size_limit = 16

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

In [3]:
df = sns.load_dataset('diamonds')
df = df[['carat', 'depth', 'table', 'price', 'x', 'y', 'z', 'cut']]

df['cut'] = (df['cut'] == 'Ideal').astype(int)

X = df.drop('cut', axis=1)
y = df['cut']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long).to(device)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.long).to(device)

print(X.shape)
print(y.shape)

ideal_percentage = df['cut'].mean() * 100
print(f"Percentage of diamonds with 'Ideal' cut: {ideal_percentage:.2f}%")

model = LogisticRegression(penalty='l2', C=0.001, max_iter=10000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

y_pred_proba = model.predict_proba(X_test)
loss = log_loss(y_test, y_pred_proba)

print(f"Test Accuracy: {accuracy * 100:.2f}%")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Not Ideal', 'Ideal']))
print(model.coef_)
print(f"Log Loss: {loss:.6f}")

(53940, 7)
(53940,)
Percentage of diamonds with 'Ideal' cut: 39.95%
Test Accuracy: 79.30%
Classification Report:
              precision    recall  f1-score   support

   Not Ideal       0.82      0.85      0.83      6496
       Ideal       0.75      0.71      0.73      4292

    accuracy                           0.79     10788
   macro avg       0.79      0.78      0.78     10788
weighted avg       0.79      0.79      0.79     10788

[[-0.15688782 -0.46189194 -1.57368125  0.22509094 -0.08309788 -0.03167662
  -0.08671039]]
Log Loss: 0.451068


/Users/nhatt/Downloads/All_Mac/Code/ML_experiments/.venv/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/nhatt/Downloads/All_Mac/Code/ML_experiments/.venv/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/nhatt/Downloads/All_Mac/Code/ML_experiments/.venv/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/nhatt/Downloads/All_Mac/Code/ML_experiments/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/nhatt/Downloads/All_Mac/Code/ML_experiments/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered

In [15]:
def train(model, criterion, optimizers, epochs):
    print(summary(model, input_data=X_train_tensor))

    torch.set_printoptions(threshold=float('inf'))
    # for name, param in model.named_parameters():
    #     print(name)
    #     print(param)

    @torch._dynamo.explain
    def run_model(x):
        return model(x)

    print(run_model(X_train_tensor)) # type: ignore

    best_test_loss = float('inf')

    train_losses = []
    test_losses = []

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

    schedulers = [
        torch.optim.lr_scheduler.LinearLR(opt, start_factor=1.0, end_factor=0.05, total_iters=epochs)
        for opt in optimizers
    ]

    for epoch in range(1, epochs + 1):
        model.train()
        for opt in optimizers:
            opt.zero_grad()

        train_outputs = model(X_train_tensor)
        train_loss = criterion(train_outputs, y_train_tensor)
        train_loss.backward()

        for opt in optimizers:
            opt.step()

        model.eval()
        with torch.no_grad():
            test_outputs = model(X_test_tensor)
            test_loss = criterion(test_outputs, y_test_tensor)
            test_predictions = torch.argmax(test_outputs, dim=1)
            test_accuracy = (test_predictions == y_test_tensor).sum().item() / y_test_tensor.size(0)

        train_losses.append(train_loss.item())
        test_losses.append(test_loss.item())

        if test_loss.item() < best_test_loss:
            best_test_loss = test_loss.item()

        for scheduler in schedulers:
            scheduler.step()

        if epoch % 1 == 0:
            print(f"Epoch {epoch}/{epochs}, Train Loss: {train_loss.item():.32f}, Test Loss: {test_loss.item():.8f}, Test Accuracy: {test_accuracy:.4f}")

    if torch.cuda.is_available():
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

        print(f"[GPU] Peak memory allocated: {torch.cuda.max_memory_allocated() / 1024 ** 2:.2f} MB")
        print(f"[GPU] Peak memory reserved: {torch.cuda.max_memory_reserved() / 1024 ** 2:.2f} MB")

    print()
    print(f"Lowest Test Loss: {best_test_loss:.8f}")

    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(test_losses, label='Test Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Test Loss Over Epochs')
    plt.legend()
    plt.grid(True)
    plt.show()  

In [5]:
class CustomActivation(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.a = nn.Parameter(torch.full((num_features,), 1.0))
        self.b = nn.Parameter(torch.full((num_features,), 1.0))

    def forward(self, x):
        return torch.where(x >= 0, self.a * x, self.b * x)

In [6]:
class MADBatchNorm1d(nn.Module):
    def __init__(self, num_features, eps=1e-5, momentum=0.1, affine=False, track_running_stats=True):
        super(MADBatchNorm1d, self).__init__()
        self.num_features = num_features
        self.eps = eps
        self.momentum = momentum
        self.track_running_stats = track_running_stats

        if self.track_running_stats:
            self.register_buffer('running_mean', torch.zeros(num_features))
            self.register_buffer('running_mad', torch.ones(num_features))
            self.running_mean: torch.Tensor
            self.running_mad: torch.Tensor
            self.register_buffer('num_batches_tracked', torch.tensor(0, dtype=torch.long))

    def forward(self, x):
        if x.dim() != 2:
            raise ValueError("Input must be 2D (batch, features)")

        if self.training or not self.track_running_stats:
            mean = x.mean(dim=0)
            mad = (x - mean).abs().mean(dim=0)

            if self.track_running_stats:
                with torch.no_grad():
                    self.running_mean.mul_(1 - self.momentum).add_(self.momentum * mean)
                    self.running_mad.mul_(1 - self.momentum).add_(self.momentum * mad)
                    self.num_batches_tracked += 1
            normed = (x - mean) / (mad + self.eps)

        else:
            normed = (x - self.running_mean) / (self.running_mad + self.eps)

        return normed

In [18]:
class StableNN(nn.Module):
    def __init__(self, n, num_layers=2):
        super().__init__()
        self.initial_norm = MADBatchNorm1d(n, affine=False, track_running_stats=True, momentum=0.0)

        self.linears = nn.ModuleList()
        self.pre_norms = nn.ModuleList()
        self.activations = nn.ModuleList()
        self.post_norms = nn.ModuleList()

        for _ in range(num_layers):
            self.linears.append(nn.Linear(n, n, bias=False))
            with torch.no_grad():
                nn.init.eye_(self.linears[-1].weight)

            self.pre_norms.append(MADBatchNorm1d(n, affine=False, track_running_stats=True, momentum=0.0))
            self.activations.append(CustomActivation(n))
            self.post_norms.append(MADBatchNorm1d(n, affine=False, track_running_stats=True, momentum=0.0))

            n *= 2

        self.final_linear = nn.Linear(n, 2, bias=True)
        with torch.no_grad():
            nn.init.zeros_(self.final_linear.weight)
            nn.init.zeros_(self.final_linear.bias)

    def forward(self, x):
        x = self.initial_norm(x)

        for linear, prenorm, activation, postnorm in zip(self.linears, self.pre_norms, self.activations, self.post_norms):
            out = linear(x)
            out = prenorm(out)
            out = activation(out)
            out = postnorm(out)

            x = torch.cat([x, out], dim=1)

        return self.final_linear(x)

In [19]:
model = (StableNN(n=X.shape[1], num_layers=8).to(device))
criterion = nn.CrossEntropyLoss().to(device)

optimizers = [
    optim.Rprop(model.parameters(), lr=0.0, step_sizes=(torch.finfo(torch.float32).smallest_normal, torch.inf))
]

train(model, criterion, optimizers, epochs=10000)

for name, param in model.named_parameters(): # type: ignore
    print(name)
    print(param)

Layer (type:depth-idx)                   Output Shape              Param #
StableNN                                 [43152, 2]                --
├─MADBatchNorm1d: 1-1                    [43152, 7]                --
├─ModuleList: 1-30                       --                        (recursive)
│    └─Linear: 2-1                       [43152, 7]                49
├─ModuleList: 1-31                       --                        --
│    └─MADBatchNorm1d: 2-2               [43152, 7]                --
├─ModuleList: 1-32                       --                        (recursive)
│    └─CustomActivation: 2-3             [43152, 7]                14
├─ModuleList: 1-33                       --                        --
│    └─MADBatchNorm1d: 2-4               [43152, 7]                --
├─ModuleList: 1-30                       --                        (recursive)
│    └─Linear: 2-5                       [43152, 14]               196
├─ModuleList: 1-31                       --              

KeyboardInterrupt: 